!pip install langchain-openai langchain openai pandas numpy matplotlib seaborn scikit-learn

In [ ]:
! wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/N0CceRlquaf9q85PK759WQ/regression-dataset.csv
! wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7J73m6Nsz-vmojwab91gMA/classification-dataset.csv

In [5]:
import numpy as np
import pandas as pd
import matplotlib
import seaborn
import sklearn
import langchain
import openai
import langchain_openai

import glob
import os
from typing import List, Optional

## Langchain tools

Tools in LangChain are interfaces that allow an AI model (such as GPT-4) to interact with external systems, retrieve data, or perform actions beyond simple text generation. These tools act as APIs or function calls that the AI can invoke when needed.

First, you'll create a tool to identify available datasets in the local directory.  This tool helps your agent discover what CSV files are available for analysis without requiring the user to explicitly specify filenames. The assumption is that the CSV files have descriptive names that indicate their content. This is typically the first step in your data analysis workflow: discovering what data is available before loading or analyzing anything.

LangChain tools are simple Python functions wrapped with the `@tool` decorator (imported from `langchain_core.tools`). They allow Large Language Models (LLMs) to call specific functions, enabling structured workflows and external tool integrations.

```python
from langchain_core.tools import tool

@tool
def my_tool(input: <type>) -> <output_type>:
    """
    Short description of what the tool does.

    Args:
        input (<type>): Explanation of the input argument.

    Returns:
        <output_type>: Explanation of the returned value.
    """
    # implement your logic here
    return tool_output
```

The tool components:

- **`@tool` Decorator:** Marks the function as a tool, allowing LangChain to integrate it and expose it to the LLM.

- **Input Arguments:** The parameters your tool function accepts, along with type annotations for clarity.

- **Tool Description:** A clear, concise explanation used by LangChain and the LLM to understand when and how to call the tool.

- **Return Type:** Specifies the type of data your tool will return, improving clarity and reliability.

- **`.name`:** Automatically derived from your Python function name; used by LangChain to identify the tool.

- **`.description`:** Automatically extracted from your function's docstring; helps the LLM understand the tool’s purpose.

- **`.args`:** Represents input arguments with their associated types, allowing LangChain to validate and pass correct values to your tool function.

Let's create the first LangChain tool, which lists all CSV files in the current directory.

- `os.getcwd()` retrieves the current working directory.
- `os.path.join(os.getcwd(), "*.csv")` constructs a path pattern to match all CSV files (`*` matches all filenames ending with `.csv`).
- `glob.glob(pattern)` returns a list of files that match the given pattern.



In [6]:
from langchain_core.tools import tool

@tool
def list_csv_files(directory: str) -> List[str]:
    """
    List all the CSV files in the specified directory.
    
    Returns:
        A list containing the names of the CSV files in the directory.
        If no csv files are found, returns an empty list.
    
    """
    csv_files = glob.glob(os.path.join(directory, "*.csv"))
    if not csv_files:
        return []
    
    return csv_files


In [7]:
list_csv_files.invoke({"directory": r"C:\\Users\\sku104\\Documents\\Projects\\Python\\LLM\\agents_step_by_step\\course6"})

['C:\\\\Users\\\\sku104\\\\Documents\\\\Projects\\\\Python\\\\LLM\\\\agents_step_by_step\\\\course6\\classification-dataset.csv',
 'C:\\\\Users\\\\sku104\\\\Documents\\\\Projects\\\\Python\\\\LLM\\\\agents_step_by_step\\\\course6\\regression-dataset.csv']

In [8]:
directory="C:\\Users\\sku104\\Documents\\Projects\\Python\\LLM\\agents_step_by_step"
glob.glob(os.path.join(directory, "*.csv"))

[]

In [9]:
from getpass import getpass
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()


openrouter_llm = ChatOpenAI(
    model="deepseek/deepseek-v4-flash",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "http://localhost",
        "X-Title": "LLM-Powered Data Science Notebook"
    },
    temperature=0
)

openrouter_llm.invoke("Reply with: OpenRouter is connected.").content

'OpenRouter is connected.'

In [ ]:
print("Tool Name:", list_csv_files.name)
print("Tool Description:", list_csv_files.description)
print("Tool Arguments:", list_csv_files.args)

### Dataset caching tool

 As you build more complex tools, you need to efficiently manage datasets in memory. Since language models communicate via text, sending entire datasets in each response would waste tokens and context window space. To solve this, you'll create a global cache that stores DataFrames after they're first loaded. This approach has several benefits:
   1. Reduces token usage by referencing datasets by name rather than content
   2. Improves performance by loading data only once
   3. Maintains dataset availability between different tool calls

 The following tool allows the agent to preload datasets into this cache system.

In [23]:

DATAFRAME_CACHE = {}

@tool
def preload_datasets(paths: List[str]) -> str:
    """
    Loads CSV files into a global cache if not loaded for quick access in future steps.

    This function helps to efficiently manage datasets by loading them once and storing them in memory for future use.

    Without caching, you would waste tokens describing dataset contents repeatedly  in agent responses.
    
    Args:
        paths: A list of file paths to CSV files.

    Returns:
        A message summarzing which datasets wehre loaded or already cached.
    """
    loaded = []
    cached = []
    for path in paths:
        if path not in DATAFRAME_CACHE:
            DATAFRAME_CACHE[path] = pd.read_csv(path)
            loaded.append(path)
        else:
            cached.append(path)
    
    return (
        f"Loaded datasets: {loaded}\n"
        f"Already cached: {cached}"
    )




In [17]:
from typing import List, Optional,Dict,Any

@tool
def get_dataset_summaries(dataset_paths: List[str]) -> List[Dict[str, Any]]:
    """
    Analyze multiple CSV files and return metadata summaries for each.

    Args:
        dataset_paths (List[str]): 
            A list of file paths to CSV datasets.

    Returns:
        List[Dict[str, Any]]: 
            A list of summaries, one per dataset, each containing:
            - "file_name": The path of the dataset file.
            - "column_names": A list of column names in the dataset.
            - "data_types": A dictionary mapping column names to their data types (as strings).
    """
    summaries = []

    for path in dataset_paths:
        # Load and cache the dataset if not already cached
        if path not in DATAFRAME_CACHE:
            DATAFRAME_CACHE[path] = pd.read_csv(path)
        
        df = DATAFRAME_CACHE[path]

        # Build summary
        summary = {
            "file_name": path,
            "column_names": df.columns.tolist(),
            "data_types": df.dtypes.astype(str).to_dict()
        }

        summaries.append(summary)

    return summaries

In [18]:
data = pd.read_csv(r"C:\Users\sku104\Documents\Projects\Python\LLM\agents_step_by_step\course6\classification-dataset.csv")

In [19]:
tst =getattr(data, "head",None)

In [20]:
@tool
def call_dataframe_method(file_name: str, method: str) -> str:
   """
   Execute a method on a DataFrame and return the result.
   This tool lets you run simple DataFrame methods like 'head', 'tail', or 'describe' 
   on a dataset that has already been loaded and cached using 'preload_datasets'.
   Args:
       file_name (str): The path or name of the dataset in the global cache.
       method (str): The name of the method to call on the DataFrame. Only no-argument 
                     methods are supported (e.g., 'head', 'describe', 'info').
   Returns:
       str: The output of the method as a formatted string, or an error message if 
            the dataset is not found or the method is invalid.
   Example:
       call_dataframe_method(file_name="data.csv", method="head")
   """
   # Try to get the DataFrame from cache, or load it if not already cached
   if file_name not in DATAFRAME_CACHE:
       try:
           DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
       except FileNotFoundError:
           return f"DataFrame '{file_name}' not found in cache or on disk."
       except Exception as e:
           return f"Error loading '{file_name}': {str(e)}"
   
   df = DATAFRAME_CACHE[file_name]
   func = getattr(df, method, None)
   if not callable(func):
       return f"'{method}' is not a valid method of DataFrame."
   try:
       result = func()
       return str(result)
   except Exception as e:
       return f"Error calling '{method}' on '{file_name}': {str(e)}"

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Assumes this global cache is shared
DATAFRAME_CACHE = {}

@tool
def evaluate_classification_dataset(file_name: str, target_column: str) -> Dict[str, float]:
    """
    Train and evaluate a classifier on a dataset using the specified target column.
    Args:
        file_name (str): The name or path of the dataset stored in DATAFRAME_CACHE.
        target_column (str): The name of the column to use as the classification target.
    Returns:
        Dict[str, float]: A dictionary with the model's accuracy score.
    """
    # Try to get the DataFrame from cache, or load it if not already cached
    if file_name not in DATAFRAME_CACHE:
        try:
            DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
        except FileNotFoundError:
            return {"error": f"DataFrame '{file_name}' not found in cache or on disk."}
        except Exception as e:
            return {"error": f"Error loading '{file_name}': {str(e)}"}
    
    df = DATAFRAME_CACHE[file_name]
    if target_column not in df.columns:
        return {"error": f"Target column '{target_column}' not found in '{file_name}'."}
    
    X = df.drop(columns=[target_column])
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestClassifier()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return {"accuracy": acc}

@tool
def evaluate_regression_dataset(file_name: str, target_column: str) -> Dict[str, float]:
    """
    Train and evaluate a regression model on a dataset using the specified target column.
    Args:
        file_name (str): The name or path of the dataset stored in DATAFRAME_CACHE.
        target_column (str): The name of the column to use as the regression target.
    Returns:
        Dict[str, float]: A dictionary with R² score and Mean Squared Error.
    """
    # Try to get the DataFrame from cache, or load it if not already cached
    if file_name not in DATAFRAME_CACHE:
        try:
            DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
        except FileNotFoundError:
            return {"error": f"DataFrame '{file_name}' not found in cache or on disk."}
        except Exception as e:
            return {"error": f"Error loading '{file_name}': {str(e)}"}
    
    df = DATAFRAME_CACHE[file_name]
    if target_column not in df.columns:
        return {"error": f"Target column '{target_column}' not found in '{file_name}'."}
    
    X = df.drop(columns=[target_column])
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestRegressor()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    return {
        "r2_score": r2,
        "mean_squared_error": mse
    }

In [12]:

openrouter_llm = ChatOpenAI(
    model="deepseek/deepseek-v4-flash",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "http://localhost",
        "X-Title": "LLM-Powered Data Science Notebook"
    },
    temperature=0,streaming=True
)

openrouter_llm.invoke("Reply with: OpenRouter is connected.").content

'OpenRouter is connected.'

In [13]:
openrouter_llm.invoke("who are you ").content

"Hi there! 👋\n\nI'm DeepSeek, an AI assistant created by the company DeepSeek (深度求索). I'm here to help you with a wide range of tasks - whether you need information, want to brainstorm ideas, need help with writing, coding, analysis, or just want to have a conversation!\n\nSome highlights about me:\n- **Knowledge cutoff**: May 2025\n- **Features**: I'm a pure text model that can read links and process uploaded files (images, PDFs, Word docs, Excel, PowerPoint, etc.)\n- **Context window**: 1 million tokens - I can process massive amounts of text at once!\n- **Free to use**: Both on web and mobile app\n- **Internet search**: Available if you manually enable it\n- **Voice input**: Supported on the app\n\nI'm designed to be helpful, detailed, and enthusiastic in my responses. So, what can I help you with today? 😊"

In [24]:
tools=[list_csv_files, preload_datasets, get_dataset_summaries, call_dataframe_method, evaluate_classification_dataset, evaluate_regression_dataset]